# Agents Functionality Test

Smoke-test every implemented specialist agent in mock mode:

- `InformationAgent`: metrics and anomaly detection
- `KnowledgeAgent`: business definitions and context, returning `data['knowledge']`
- `MetadataAgent`: ownership, DQ scores, lineage
- `CapacityAgent`: Jira issue read and ticket creation
- `RuleAgent`: list, create, and evaluate rules; evaluate routing is checked before list routing

In [3]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / 'src').exists()), cwd)
sys.path.insert(0, str(project_root / 'src'))
print('project_root:', project_root)

project_root: D:\0_PROJECTS\data-governance-copilot


In [4]:
from config.settings import AppConfig
from core.base_agent import AgentRequest
from agents.information_agent import InformationAgent
from agents.knowledge_agent import KnowledgeAgent
from agents.metadata_agent import MetadataAgent
from agents.capacity_agent import CapacityAgent
from agents.rule_agent import RuleAgent, RULE_REGISTRY

config = AppConfig()
agents = {
    'information': InformationAgent(config=config, enable_mock=True),
    'knowledge': KnowledgeAgent(config=config, enable_mock=True),
    'metadata': MetadataAgent(config=config, enable_mock=True),
    'capacity': CapacityAgent(config=config, enable_mock=True),
    'rule': RuleAgent(config=config, enable_mock=True),
}

for name, agent in agents.items():
    print(name, agent.health_check())

information {'agent': 'information_agent', 'healthy': True, 'mock_mode': True, 'capabilities': ['metric_retrieval', 'trend_comparison', 'anomaly_detection', 'dimensional_analysis']}
knowledge {'agent': 'knowledge_agent', 'healthy': True, 'mock_mode': True, 'capabilities': ['business_definitions', 'contextual_explanation', 'runbook_retrieval']}
metadata {'agent': 'metadata_agent', 'healthy': True, 'mock_mode': True, 'capabilities': ['data_quality_scores', 'ownership_stewardship', 'lineage_tracing', 'classification_retrieval']}
capacity {'agent': 'capacity_agent', 'healthy': True, 'mock_mode': True, 'capabilities': ['issue_retrieval', 'incident_tracking', 'blocker_identification', 'ticket_creation']}
rule {'agent': 'rule_agent', 'healthy': True, 'mock_mode': True, 'capabilities': ['rule_creation', 'rule_listing', 'rule_evaluation', 'dq_rule_management', 'business_rule_management']}


In [5]:
def run_agent(name, query, intent='', products=None, context=None):
    request = AgentRequest(
        query=query,
        intent=intent,
        data_products=products or [],
        context=context or {},
        time_range='last_month',
    )
    result = agents[name].execute(request)
    print('\n' + '=' * 80)
    print(name.upper(), 'success=', result.success, 'confidence=', result.confidence)
    print('sources:', result.sources)
    print('metadata:', result.metadata)
    print(result.summary[:1200])
    assert result.success
    return result

info = run_agent('information', 'Why did retention drop last month?', products=['retention'])
knowledge = run_agent('knowledge', 'What is GRR and how is it calculated?')
assert 'knowledge' in knowledge.data
print('knowledge entries:', len(knowledge.data['knowledge']))
metadata = run_agent('metadata', 'Who owns the retention dataset?', products=['retention'])
capacity_read = run_agent('capacity', 'Show open Jira bugs for retention', products=['retention'])
rule_list = run_agent('rule', 'List all data quality rules')

[10:44:59] INFO     agent.information_agent | [e6146ef8] information_agent received: 'Why did retention drop last month?'
[10:44:59] INFO     agent.information_agent | [e6146ef8] information_agent done in 4.0ms

INFORMATION success= True confidence= 0.85
sources: ['[MOCK] analytics.retention_metrics']
metadata: {'products_queried': ['retention'], 'time_range': 'last_month'}
📊 **Metrics Summary** (last_month)

**RETENTION**
  • Gross Retention Rate: 84.79
  • Gross Retention Prev: 87.3
  • Net Retention Rate: 107.82
  • Churn Rate: 15.21
  • Churned Accounts: 35
  • Total Accounts: 412
  • At Risk Accounts: 24

**🚨 Anomalies:**
  ⚠️ GRR (84.79%) below 85% threshold — investigate churn drivers immediately.
[10:44:59] INFO     agent.knowledge_agent | [815d2ee7] knowledge_agent received: 'What is GRR and how is it calculated?'
[10:44:59] INFO     agent.knowledge_agent | [815d2ee7] knowledge_agent done in 2.2ms

KNOWLEDGE success= True confidence= 0.82
sources: ['SharePoint: CS Strategy Dec

In [6]:
# Capacity write path: mock Jira ticket creation.
ticket = run_agent(
    'capacity',
    'Create ticket for retention completeness issue',
    products=['retention'],
    context={
        'ticket_summary': 'Notebook test ticket',
        'ticket_description': 'Created by agents functionality notebook.',
        'priority': 'High',
    },
)
print('ticket_id:', ticket.data.get('ticket_id'))
assert ticket.data.get('ticket_id')

[10:45:54] INFO     agent.capacity_agent | [ebea892d] capacity_agent received: 'Create ticket for retention completeness issue'
[10:45:54] INFO     agent.capacity_agent | [ebea892d] capacity_agent done in 2.0ms

CAPACITY success= True confidence= 0.9
sources: ['Jira (Mock)']
metadata: {}
✅ [MOCK] Jira Bug created: **DATA-5781**
   Notebook test ticket
ticket_id: DATA-5781


In [7]:
# Rule create and evaluate paths.
before = len(RULE_REGISTRY)
created = run_agent(
    'rule',
    'Create rule for retention freshness threshold',
    products=['retention'],
    context={
        'rule_name': 'Notebook Retention Freshness Check',
        'asset': 'analytics.retention_metrics',
        'expression': 'last_refresh_hours < 24',
        'threshold': 24,
        'severity': 'Medium',
    },
)
after = len(RULE_REGISTRY)
print('rules before:', before, 'after:', after)
assert after == before + 1

evaluation = run_agent('rule', 'Evaluate all retention rules', products=['retention'])
assert isinstance(evaluation.data, list)
assert 'passed' in evaluation.metadata and 'failed' in evaluation.metadata
assert 'Rule Evaluation' in evaluation.summary
print('evaluation metadata:', evaluation.metadata)

[10:46:43] INFO     agent.rule_agent | [46b5d598] rule_agent received: 'Create rule for retention freshness threshold'
[10:46:43] INFO     agent.rule_agent | Rule created: BR-AB68 (Notebook Retention Freshness Check)
[10:46:43] INFO     agent.rule_agent | [46b5d598] rule_agent done in 6.02ms

RULE success= True confidence= 1.0
sources: ['Rule Registry']
metadata: {'rule_id': 'BR-AB68', 'rule_type': 'business_rule'}
✅ Rule **BR-AB68** created: _Notebook Retention Freshness Check_
  Type: business_rule | Severity: Medium | Asset: analytics.retention_metrics
rules before: 4 after: 5
[10:46:43] INFO     agent.rule_agent | [63e1acfc] rule_agent received: 'Evaluate all retention rules'
[10:46:43] INFO     agent.rule_agent | [63e1acfc] rule_agent done in 2.51ms

RULE success= True confidence= 0.85
sources: ['Rule Registry']
metadata: {'passed': 2, 'failed': 2}
📋 **Rule Evaluation** (4 rules)
  ❌ **DQ-001**: Retention Completeness Check
  ✅ **DQ-002**: Retention Rate Range Validity
  ✅ **BR-00